In [10]:
import os
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [11]:
from src.utils.transforming import basic_transformation

transform = basic_transformation

In [12]:

class TestDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.image_files = sorted([f for f in os.listdir(image_dir) if f.endswith('.jpg')])
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, img_name

In [ ]:
test_images_dir = "TEST_IMAGES_DIRECTORY"

dataset = TestDataset(image_dir=test_images_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

num_classes = 20
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from src.models.__all_models import PoseCNNsc
model = PoseCNNsc(num_classes=num_classes)

PATH_TO_WEIGHTS = "PATH_TO_YOUR_WEIGHTS"
model.load_state_dict(torch.load(
    PATH_TO_WEIGHTS,
    map_location=device
))


<All keys matched successfully>

In [ ]:
model.to(device)
model.eval()


all_preds = []
all_img_names = []

with torch.no_grad():
    for images, img_names in tqdm(dataloader):
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().tolist())
        all_img_names.extend(img_names)


results_df = pd.DataFrame({
    'img_name': all_img_names,
    'predicted': all_preds 
})
results_df.to_csv('inference_results_notebook.csv', index=False)
print(f"Results saved!")

100%|██████████| 1/1 [00:01<00:00,  1.39s/it]

Results saved!
